# π on the GPU — but only if you feed it.  ~5 min

The course's red thread ends here: we estimated π by Monte-Carlo **sequentially** (module 1), then across **threads** (module 3); now on the **GPU**.

**Needs a GPU.** On Colab: *Runtime ▸ Change runtime type ▸ GPU* (free). Fill one blank, run, read the table. We write **no CUDA kernel** — `torch` runs the same array code on the card.

In [ ]:
import torch, time
assert torch.cuda.is_available(), 'No GPU — on Colab: Runtime ▸ Change runtime type ▸ GPU'
print('GPU:', torch.cuda.get_device_name(0))

def pi_cpu(n):
    x = torch.rand(n); y = torch.rand(n)          # n random points in the unit square
    return 4 * ((x*x + y*y) <= 1.0).sum().item() / n   # 4 × (fraction inside the quarter disk)

**Your turn.** Write `pi_gpu`: the *same* computation, but with the random draws made **on the GPU** (`torch.rand(n, device='cuda')`). Nothing else changes — no kernel.

In [ ]:
def pi_gpu(n):
    ## SOLUTION: draw x and y with device='cuda', same formula, then torch.cuda.synchronize() ##
    x = torch.rand(n, device='cuda'); y = torch.rand(n, device='cuda')
    hits = ((x*x + y*y) <= 1.0).sum()
    torch.cuda.synchronize()          # wait for the GPU to actually finish before we time it
    return 4 * hits.item() / n
    ## END ##

In [ ]:
# Run me — the same π, three problem sizes, CPU vs GPU
def best(f, n, iters=10):
    f(n)                              # warm up (1st GPU call compiles / allocates)
    t0 = time.perf_counter()
    for _ in range(iters): p = f(n)
    return (time.perf_counter() - t0) / iters, p

print(f"{'points':>12} | {'CPU':>9} | {'GPU':>9} | speedup")
for n in (10_000, 1_000_000, 50_000_000):
    tc, _ = best(pi_cpu, n)
    tg, p = best(pi_gpu, n)
    print(f'{n:>12,} | {tc*1e3:7.2f}ms | {tg*1e3:7.2f}ms | {tc/tg:5.1f}×   (π≈{p:.4f})')

### What you just saw

At **10 000** points the GPU is **~1× or slower** — you paid to ship data to the card and launch it, for almost no work. At **millions** of points it's tens of ×. A GPU has thousands of tiny cores; they only pay off when there's enough work to keep them all busy. **You must feed the GPU.**

> No kernel here — the *same* `torch.rand` and arithmetic as the CPU version; `torch` dispatched it onto the GPU because the tensors live there. (Writing your own kernel is a separate, deeper topic.) And the red thread is complete: **π — sequential → threads → GPU.**